In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys

import torch

from spice import SpiceEstimator

sys.path.append("../../..")
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, get_dataset, generate_behavior
import spice_castro2025, spice_castro2025_2

## Load dataset

In [3]:
path_data = 'data/eckstein2024.csv'
test_sessions = (1, 3)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4


In [ ]:
# keep only 100 participants for rapid prototyping
from spice import SpiceDataset

keep_participants = torch.arange(0, 100)

mask_train = torch.isin(dataset_train.xs[..., -1], keep_participants)
mask_test = torch.isin(dataset_test.xs[..., -1], keep_participants)

xs, ys = dataset_train.xs[mask_train], dataset_train.ys[mask_train]
dataset_train = SpiceDataset(xs, ys)

xs, ys = dataset_test.xs[mask_test], dataset_test.ys[mask_test]
dataset_test = SpiceDataset(xs, ys)


## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
from spice import SpiceConfig


CONFIG = SpiceConfig(
    library_setup={
        'value_reward_env': [
            'reward',
        ],
        'value_reward_chosen': [
            'reward_env',
            'reward',
            'dvalue',
            'd²value',
        ],
        'value_reward_not_chosen': [
            'reward_env',
            'dvalue',
            'd²value',
        ],
        'value_choice_chosen': [
            'dvalue',
            'd²value',
        ],
        'value_choice_not_chosen': [
            'dvalue',
            'd²value',
        ],
    },
    memory_state={
        'value_reward_env': 0.,
        
        'value_reward': 0.,
        'dvalue_reward': 0.,
        'd²value_reward': 0.,
        'buffer_value_reward': 0,
        'buffer_dvalue_reward': 0.,
        
        'value_choice': 0.,
        'dvalue_choice': 0.,
        'd²value_choice': 0.,
        'buffer_value_choice': 0,
        'buffer_dvalue_choice': 0.,
    },
    states_in_logit=['value_reward', 'value_choice'],
)

In [ ]:
from spice import BaseModel


class SpiceModel(BaseModel):
    """
    Value learning with delta-value working memory buffers.
    
    Buffers store recent value changes (dV) rather than raw observations or values.
    The current state encodes the level; dV encodes the recent trajectory
    (trend, volatility), providing non-redundant temporal information.
    """
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        dropout = 0.1
        
        self.participant_embedding = self.setup_embedding(self.n_participants, self.embedding_size, dropout=dropout)
        
        self.setup_module(key_module='value_reward_env', input_size=1+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_reward_chosen', input_size=4+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_reward_not_chosen', input_size=3+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_choice_chosen', input_size=2+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_choice_not_chosen', input_size=2+self.embedding_size, dropout=dropout)
        
    def forward(self, inputs, state=None):
        spice_signals = self.init_forward_pass(inputs, state)

        reward_full = spice_signals.rewards.sum(dim=-1, keepdim=True).expand_as(spice_signals.actions)
        
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)

        for trial in spice_signals.trials:
            
            # Snapshot pre-update values for delta computation
            self.state['buffer_value_reward'] = self.get_state()['value_reward']
            self.state['buffer_value_choice'] = self.get_state()['value_choice']
            self.state['buffer_dvalue_reward'] = self.get_state()['dvalue_reward']
            self.state['buffer_dvalue_choice'] = self.get_state()['dvalue_choice']
            
            # REWARD VALUE UPDATES
            value_reward_env = self.call_module(
                key_module='value_reward_env',
                key_state='value_reward_env',
                inputs=reward_full[trial],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            self.call_module(
                key_module='value_reward_chosen',
                key_state='value_reward',
                action_mask=spice_signals.actions[trial],
                inputs=(
                    value_reward_env,
                    spice_signals.rewards[trial],
                    self.state['dvalue_reward'],
                    self.state['d²value_reward'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )

            self.call_module(
                key_module='value_reward_not_chosen',
                key_state='value_reward',
                action_mask=1-spice_signals.actions[trial],
                inputs=(
                    value_reward_env,
                    self.state['dvalue_reward'],
                    self.state['d²value_reward'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )

            # CHOICE VALUE UPDATES
            self.call_module(
                key_module='value_choice_chosen',
                key_state='value_choice',
                action_mask=spice_signals.actions[trial],
                inputs=(
                    self.state['dvalue_choice'],
                    self.state['d²value_choice'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            self.call_module(
                key_module='value_choice_not_chosen',
                key_state='value_choice',
                action_mask=1-spice_signals.actions[trial],
                inputs=(
                    self.state['dvalue_choice'],
                    self.state['d²value_choice'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            # BUFFER UPDATES: compute deltas and shift
            self.state['dvalue_reward'] = self.state['value_reward'] - self.state['buffer_value_reward']
            self.state['dvalue_choice'] = self.state['value_choice'] - self.state['buffer_value_choice']
            self.state['d²value_reward'] = self.state['dvalue_reward'] - self.state['buffer_dvalue_reward']
            self.state['d²value_choice'] = self.state['dvalue_choice'] - self.state['buffer_dvalue_choice']

            # Logits
            spice_signals.logits[trial] = (
                self.state['value_reward'] 
                + self.state['value_choice']
            )

        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

In [ ]:
# fitting without SINDy coefficients
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=SpiceModel,
        spice_config=CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        sindy_alpha=0.0001,
        
        save_path_spice=path_spice,
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [4]:
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025.SpiceModel,
        spice_config=spice_castro2025.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=10,
        warmup_steps=200,
        learning_rate=0.01,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,

        compiled_forward=False,
        verbose=True,
        # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [5]:
estimator.load_spice(path_spice)

In [6]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=5)


Example SPICE model (participant 0):
value_reward_env[t+1]        = 1.0 value_reward_env[t] 
value_reward_chosen[t+1]     = -1.618 1 + 1.0 value_reward_chosen[t] + 1.758 reward + 1.316 reward^2 
value_reward_not_chosen[t+1] = 1.0 value_reward_not_chosen[t] 
value_choice_chosen[t+1]     = 1.0 value_choice_chosen[t] 
value_choice_not_chosen[t+1] = 1.0 value_choice_not_chosen[t] 


In [7]:
estimator.aggregate_coefficients()

## Benchmarking

### Castro2025 benchmark model

In [8]:
# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=dataset_train.n_participants,
    n_actions=dataset_train.n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [ ]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
)

torch.save(cfs.state_dict(), path_cfs)

In [9]:
cfs.load_state_dict(torch.load(path_cfs, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [10]:
gru = GRUModel(
    n_actions=dataset_train.n_actions, 
    additional_inputs=2, 
    dropout=0.1,
    embedding_size=16,
    hidden_size=32,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [ ]:
epochs = 2000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

In [11]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [12]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from analysis_generative import analysis_generative_behavior

## General Analysis

In [13]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [14]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

Fitted Castro2025 parameters:

Beta_r
tensor([3.0227, 2.7835, 3.8847, 2.1398, 2.6095, 2.0641, 2.2355, 2.4648, 1.6898,
        2.5949, 2.5302, 1.8831, 2.5011, 1.1640, 2.3157, 1.9224, 2.6862, 2.1197,
        2.6740, 3.1597, 2.1158, 2.5111, 1.9301, 3.0150, 2.2456, 1.9637, 1.8244,
        2.0220, 2.4595, 2.2567, 2.4239, 2.2062, 2.1208, 2.9200, 2.1419, 2.4809,
        2.3059, 2.0881, 3.4308, 3.4039, 1.8621, 1.7212, 2.5857, 1.6436, 2.2871,
        2.6510, 2.3212, 1.7566, 2.1336, 2.5991, 2.7140, 1.9370, 1.9595, 1.6267,
        1.8756, 2.4850, 2.0949, 2.2517, 2.4824, 2.5229, 1.7200, 2.1338, 2.6180,
        1.8628, 2.3951, 1.9519, 2.3674, 2.3258, 1.5663, 1.9482, 2.8100, 1.7558,
        2.4880, 3.1862, 1.9455, 3.2004, 2.8010, 1.5734, 2.0922, 1.7111, 2.6521,
        2.0788, 3.0228, 1.9182, 1.7887, 1.9857, 1.7441, 2.5801, 1.5778, 2.6975,
        1.8457, 2.5325, 1.7934, 2.9219, 1.9543, 1.1464, 2.8707, 3.6663, 2.1621,
        2.1932, 2.0553, 2.0168, 2.1864, 2.4884, 2.5875, 2.5489, 1.8931, 1.9988,
  

## Analysis Model Evaluation

In [16]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.569498,0.167914,13.000000,0.00000,143885.937500,2.877979e+05,2.879338e+05
GRU,0.602542,0.160125,6852.000000,0.00000,129471.273438,2.726466e+05,3.442585e+05
SPICE-RNN,0.267789,0.217929,79422.000000,0.00000,336727.312500,8.322986e+05,1.662358e+06
SPICE,0.052892,0.162571,5.016241,2.49986,751248.187500,1.502506e+06,1.502559e+06


## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

## Generative Behavior Analysis

In [ ]:
estimator.eval()
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice.csv'
)

estimator.use_sindy(False)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice_rnn.csv'
)

gru.eval()
generate_behavior(
    model=gru,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_gru.csv'
)

generate_behavior(
    model=cfs,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_cfs.csv'
)

In [ ]:
analysis_generative_behavior(
    path_data_real=path_data,
    path_data_gru='data/eckstein2024_gru.csv',
    path_data_benchmark='data/eckstein2024_cfs.csv',
    path_data_spice='data/eckstein2024_spice.csv',
    path_data_spice_rnn='data/eckstein2024_spice_rnn.csv',
    output_dir='results',
)